ForEachBatch :
- Source Allows us to process data in batches : Micro Batch
- On top of this micro batch we can apply any function.
- Thats why it is called ForEachBatch

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.readStream.table('dbcertification.streaming.sourcetable')

In [0]:
def myfunc(df,batchid):

    df = df.groupBy("color").agg(count("*").alias("count"))

    # destination 1
    (df.write.format("delta")
        .mode("append")
        .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/foreach/sinktable1/")
        .save() 
    )

    #destination 2

    (df.write.format("delta")
        .mode("append")
        .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/foreach/sinktable2/")
        .save() 
    )

In [0]:
(
df.writeStream.foreachBatch(myfunc)
    .outputMode("append")
    .trigger(once=True)
    .option("checkpointLocation","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/foreach/checkpointLocation")
    .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/foreach/dummyPath/")
    .start()
)

- I can simply write non-streaming code in foreach batch function and simply run it as a stream function.
- Applying merge is a common usecase

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/foreach/sinktable1/`

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/foreach/sinktable2/`